# Prior elicitation for the initial state

Does the structured `phi_init` prior help or hurt?

On real data we cannot tell: a posterior that agrees with the prior is equally consistent
with "the prior is right" and "the prior won". Here we generate synthetic data from a
**known** initial state, fit it under priors of varying strength, and measure whether the
truth is recovered.

`PHI_CONC_AMP` controls the prior: `conc = PHI_CONC_BASE + PHI_CONC_AMP * shape`.
`0` is a flat Dirichlet, `4` is what `d_45` / `d_46` used.
It is read as a module global at call time, so rebinding `ca.PHI_CONC_AMP` is enough -
no change to the package.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cme.decision_models.confidence_accumulation as ca
from numpyro.diagnostics import summary

I, J = 5, 30
N_STATES, RESPONSE_WIDTH = 51, 17
DELTA, MEASUREMENT_PROB = 0.1, 0.2
PHI_INIT_BINS = 5
NUM_WARMUP, SAMPLES_N, NUM_CHAINS = 500, 500, 4
N_FREE = N_STATES - 2 * RESPONSE_WIDTH

print("free states:", N_FREE, " bins:", PHI_INIT_BINS,
      " current PHI_CONC_AMP:", ca.PHI_CONC_AMP, " PHI_CONC_BASE:", ca.PHI_CONC_BASE)

## Helpers

In [ ]:
def true_phi(shape, n_states=N_STATES, rw=RESPONSE_WIDTH):
    """Initial state used to GENERATE the data. Returns a full n_states vector."""
    n_free = n_states - 2 * rw
    x = np.arange(n_free) - (n_free - 1) / 2
    w = n_free / 6
    if shape == "centre":
        p = np.exp(-0.5 * (x / w) ** 2)
    elif shape == "edge":
        e = (n_free - 1) / 2
        p = np.exp(-0.5 * ((x - e) / w) ** 2) + np.exp(-0.5 * ((x + e) / w) ** 2)
    elif shape == "flat":
        p = np.ones(n_free)
    else:
        raise Exception("shape must be centre, edge or flat")
    full = np.zeros(n_states)
    full[rw:rw + n_free] = p / p.sum()
    return full


def bin_masses(per_state_free, n_bins):
    """Collapse per-state values over the free region into the n_bins the Dirichlet samples."""
    sizes = [len(s) for s in np.array_split(np.arange(len(per_state_free)), n_bins)]
    out, i = [], 0
    for s in sizes:
        out.append(per_state_free[i:i + s].sum())
        i += s
    return np.array(out)


def tv(a, b):
    return 0.5 * np.abs(np.asarray(a) - np.asarray(b)).sum()

In [ ]:
def make_data(shape, model_type, I=I, J=J, seed=0, n_states=N_STATES, rw=RESPONSE_WIDTH):
    """Generate RT and RA from a known initial state."""
    full = true_phi(shape, n_states, rw)
    phi = np.tile(full, (I, 1, 1, 1)).transpose(0, 1, 3, 2)
    if model_type == "Quantum":
        phi = phi ** 0.5
    mu = np.full((I, 1), 0.5)
    sig = np.full((I, 1), 1.0)
    X = np.random.default_rng(seed).binomial(1, 0.5, (I, J))
    out = ca.get_RT(None, n_states, rw, DELTA, MEASUREMENT_PROB, X, mu, sig, phi,
                    data_samples=(I, J), min_RT_sec=0.3, max_RT_sec=8.0, param_sample_id=0,
                    model_type=model_type, transition_type="RT", likelihood_type="SINGLE",
                    sampling_type="GEN")
    s = out["Samples"].reset_index().sort_values("part_id", kind="stable")
    return s["RT"].to_numpy().reshape(I, J), s["RA"].to_numpy().reshape(I, J), full

In [ ]:
def run_config(amp, shape, model_type, I=I, J=J, n_states=N_STATES, rw=RESPONSE_WIDTH,
               n_bins=PHI_INIT_BINS, seed=0, num_warmup=None, samples_n=None):
    """Generate from `shape`, fit under prior strength `amp`, return recovery metrics."""
    num_warmup = NUM_WARMUP if num_warmup is None else num_warmup
    samples_n = SAMPLES_N if samples_n is None else samples_n
    n_free = n_states - 2 * rw
    RT, RA, truth_full = make_data(shape, model_type, I, J, seed, n_states, rw)

    old_amp = ca.PHI_CONC_AMP
    ca.PHI_CONC_AMP = amp
    try:
        conc = np.asarray(ca._initial_state_concentration(n_bins, model_type))
        t0 = time.perf_counter()
        chain = ca.sample_posterior_params(
            RT, RA, n_states=n_states, start_width=None, response_width=rw,
            delta=DELTA, measurement_prob=MEASUREMENT_PROB,
            num_warmup=num_warmup, samples_n=samples_n, num_chains=NUM_CHAINS,
            params_type="Centralized", model_type=model_type,
            transition_type="RT", likelihood_type="SINGLE", phi_init_bins=n_bins)
        secs = time.perf_counter() - t0
    finally:
        ca.PHI_CONC_AMP = old_amp

    post = chain.get_samples()
    extra = chain.get_extra_fields()
    stats = summary(chain.get_samples(group_by_chain=True), prob=0.9)

    prior_bins = conc / conc.sum()
    true_bins = bin_masses(truth_full[rw:rw + n_free], n_bins)
    phi_post = np.atleast_2d(np.asarray(post["phi_init"]).mean(axis=0).squeeze())
    post_bins = np.stack([bin_masses(p, n_bins) for p in phi_post]).mean(axis=0)

    return dict(
        amp=amp, truth=shape, model=model_type, I=I, J=J, n_states=n_states,
        recovery=tv(post_bins, true_bins),          # posterior vs TRUTH  <-- the key number
        prior_err=tv(prior_bins, true_bins),        # how wrong the prior was to begin with
        moved=tv(post_bins, prior_bins),            # how far the data pulled it
        lr=post_bins[-1] / post_bins[0],
        mid=post_bins[len(post_bins) // 2],
        rhat_phi=float(np.nanmax(stats["phi_init"]["r_hat"])),
        rhat_likl=float(np.nanmax(stats["likl_total"]["r_hat"])),
        steps=float(np.asarray(extra["num_steps"]).mean()),
        divergences=int(np.asarray(extra["diverging"]).sum()),
        secs=secs,
        prior_bins=np.round(prior_bins, 3), post_bins=np.round(post_bins, 3),
        true_bins=np.round(true_bins, 3),
    )

## Sanity check: what the priors and the truths look like

Before fitting anything, confirm the prior shape changes with `PHI_CONC_AMP` as intended
and that the two generating truths are genuinely different.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 3.5))

old = ca.PHI_CONC_AMP
for mt, ax in zip(["Markov", "Quantum"], axs[:2]):
    for amp in [0, 1, 2, 4, 8]:
        ca.PHI_CONC_AMP = amp
        conc = np.asarray(ca._initial_state_concentration(PHI_INIT_BINS, mt))
        ax.plot(conc / conc.sum(), marker="o", label=f"AMP={amp}")
    ax.set_title(f"{mt} prior mean, per bin")
    ax.set_xlabel("bin")
    ax.legend(fontsize=7)
ca.PHI_CONC_AMP = old

for shape in ["centre", "edge", "flat"]:
    axs[2].plot(true_phi(shape), label=shape)
axs[2].set_title("generating initial states (full state space)")
axs[2].set_xlabel("state")
axs[2].legend(fontsize=7)
plt.tight_layout()

## Sweep

`recovery` is the total-variation distance between the posterior and the **true** initial
state that generated the data. Lower is better. `prior_err` is how far the prior itself
sits from the truth, so the informative rows are those where `prior_err` is large.

In [ ]:
AMPS = [0, 1, 2, 4, 8]
SHAPES = ["centre", "edge"]
MODELS = ["Markov", "Quantum"]

rows = []
for model_type in MODELS:
    for shape in SHAPES:
        for amp in AMPS:
            r = run_config(amp, shape, model_type)
            rows.append(r)
            print(f"{model_type:8s} truth={shape:6s} AMP={amp}  "
                  f"recovery={r['recovery']:.3f}  prior_err={r['prior_err']:.3f}  "
                  f"rhat={r['rhat_phi']:.2f}  div={r['divergences']}  {r['secs']:.0f}s")

df = pd.DataFrame(rows)
df.to_csv("export/prior_elicitation_sweep.csv", index=False)
df[["model", "truth", "amp", "recovery", "prior_err", "moved", "lr", "mid",
    "rhat_phi", "rhat_likl", "steps", "divergences", "secs"]]

In [ ]:
fig, axs = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 4), squeeze=False)
for ax, model_type in zip(axs[0], MODELS):
    sub = df.query("model == @model_type")
    for shape in SHAPES:
        s = sub.query("truth == @shape").sort_values("amp")
        ax.plot(s.amp, s.recovery, marker="o", label=f"truth = {shape}")
    ax.axvline(4, color="grey", ls="--", lw=1)
    ax.set_title(model_type)
    ax.set_xlabel("PHI_CONC_AMP  (0 = flat prior, 4 = d_45/d_46)")
    ax.set_ylabel("recovery error (TV from truth)")
    ax.legend()
plt.tight_layout()

## How much data is needed to overcome the prior

The Quantum concentration at `AMP=4` totals about 17 pseudo-counts. Against 30 trials that
is comparable weight. This sweep shows where the data starts to win.

In [ ]:
J_VALUES = [10, 30, 100]
rows_j = []
for model_type in MODELS:
    for shape in SHAPES:
        for amp in [0, 4]:
            for Jv in J_VALUES:
                r = run_config(amp, shape, model_type, J=Jv)
                rows_j.append(r)
                print(f"{model_type:8s} truth={shape:6s} AMP={amp} J={Jv:3d}  "
                      f"recovery={r['recovery']:.3f}  {r['secs']:.0f}s")

df_j = pd.DataFrame(rows_j)
df_j.to_csv("export/prior_elicitation_trials.csv", index=False)
df_j[["model", "truth", "amp", "J", "recovery", "prior_err", "moved", "rhat_phi", "divergences"]]

In [ ]:
fig, axs = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 4), squeeze=False)
for ax, model_type in zip(axs[0], MODELS):
    for shape in SHAPES:
        for amp in [0, 4]:
            s = df_j.query("model == @model_type and truth == @shape and amp == @amp").sort_values("J")
            ax.plot(s.J, s.recovery, marker="o",
                    ls="-" if amp == 0 else "--", label=f"{shape}, AMP={amp}")
    ax.set_xscale("log")
    ax.set_title(model_type)
    ax.set_xlabel("trials per participant")
    ax.set_ylabel("recovery error (TV from truth)")
    ax.legend(fontsize=8)
plt.tight_layout()

## Reading the result

- `recovery` **flat across AMP** - the prior costs nothing; keep it.
- `recovery` **rising with AMP where `prior_err` is large** - the prior is preventing the
  model from finding a truth it disagrees with. That is the case to worry about, and the fix
  is to lower `PHI_CONC_AMP` rather than drop the shape.
- `recovery` **falling with AMP** - the prior is regularising a weakly identified parameter
  and earning its place.
- Always read it against `divergences` and `rhat_phi`: a low `AMP` that recovers well but
  does not converge is not usable.